# The polyalanine figure, end to end

This notebook runs `schamp`'s whole pipeline -- extraction, fitting, regression, cross
sections -- on the poly-DL-alanine drift-voltage series of Allen, Giles, Gilbert & Bush,
*Analyst* 2016, **141**, 884 (`docs/2016-allen-rf-confining-drift-cell.pdf`, CC-BY), and
draws the paper's Figure 1: a mobiligram heat map, drift time against reciprocal drift
voltage per oligomer, and collision cross section against m/z for all four ion series.

**The acquisitions are not shipped with this repository.** The series is two sets of 14
Waters `.raw` directories, about 5.4 GB, acquired 2013-04-23. `examples/data/` carries
only what describes the series -- the pressures and temperatures of
`examples/data/{npalan,palan}/conditions.csv` -- and this notebook resolves the
acquisitions themselves at run time, from whichever of these two places has them:

1. `$SCHAMP_EXAMPLE_RAW`, a directory holding `130423_SJA_NPALAN_1_001.raw` .. `_014.raw`
   and `130423_SJA_PALAN_1_001.raw` .. `_014.raw` directly;
2. a private lab checkout that ships them, through `schamp.lab_dir("legacy")`.

Point this notebook at a series of your own by writing a `conditions.csv` in the same
shape (see `examples/README.md`) and editing the species definitions in the extraction
section's first code cell. Nothing else changes: the pipeline calls only
`schamp.experiment`, `schamp.extract`, `schamp.atd`, `schamp.mobility` and
`schamp.report`, none of which knows anything about polyalanine.

In [ ]:
import os

import numpy as np
from IPython.display import Markdown, display

import schamp
from schamp import atd, calibration, extract, mobility, report, sdk

HERE = os.path.join(schamp.ROOT, "examples")
DATA_DIR = os.path.join(HERE, "data")
OUT_DIR = os.path.join(HERE, "out")
os.makedirs(OUT_DIR, exist_ok=True)

PROFILE = schamp.load_profile("uw-synapt-g2")


def resolve_raw_root():
    """Where the 130423 acquisitions live on this machine, or a clear error."""
    env = os.environ.get("SCHAMP_EXAMPLE_RAW")
    if env and os.path.isdir(env):
        return env
    legacy = schamp.lab_dir("legacy")
    if legacy:
        candidate = os.path.join(legacy, "130423")
        if os.path.isdir(candidate):
            return candidate
    raise RuntimeError(
        "No polyalanine acquisitions found. Set SCHAMP_EXAMPLE_RAW to a directory "
        "holding 130423_SJA_NPALAN_1_001.raw .. _014.raw and "
        "130423_SJA_PALAN_1_001.raw .. _014.raw, or run from a lab checkout that ships "
        "them. See examples/README.md."
    )


RAW_ROOT = resolve_raw_root()
print(f"acquisitions: {RAW_ROOT}")

In [ ]:
def show_table(header, rows, *, title="", limit=8):
    """A quick markdown table for a notebook cell -- not schamp's own writer, which
    goes to a file (`report.write_table`, used below for the tables that matter)."""
    lines = [f"**{title}**", ""] if title else []
    lines.append("| " + " | ".join(header) + " |")
    lines.append("|" + "|".join(["---"] * len(header)) + "|")
    for row in rows[:limit]:
        lines.append("| " + " | ".join(str(v) for v in row) + " |")
    if len(rows) > limit:
        lines.append(f"| ... | {len(rows) - limit} more row(s) | " + " | ".join([""] * (len(header) - 2)))
    display(Markdown("\n".join(lines)))

## 1. Extraction: a `.raw` plus m/z windows, to arrival-time distributions

Each acquisition is loaded through `schamp.experiment`, which reads its own conditions
table and, from `_extern.inf`, its own drift voltage -- nothing is typed in here. A
window is placed on every oligomer's own measured peak in the acquisition's total mass
spectrum (`extract.MzWindow.around_measured`), three measured peak widths across, and
converted into the frame `ReadMobillogram` is addressed in by the acquisition's own
`_HEADER.TXT` calibration function (`calibration.MzFrame`) -- the placement a lab
checkout of this project settled on precisely because a width chosen in Th, rather than
in peak widths, is a different number of peaks at 230 Th and at 1650 Th
(`notes/extraction.md`, lab record). A species too weak to measure falls back to a fixed
width on its computed m/z, and says so; on this series that fallback is never taken.

In [ ]:
RESIDUE_DA = 71.03711   # alanine residue, monoisotopic
WATER_DA = 18.010565    # terminal water
PROTON_DA = 1.007276

FALLBACK_WIDTH_TH = 0.5  # only used when a species has no measurable peak
MIN_APEX_COUNTS = 2.0e4  # the summed baseline of these acquisitions is around 1.4e3

# One entry per ion series: which acquisitions, which charge states, which oligomers.
SERIES = {
    "NPALAN": {
        "title": "130423 poly-DL-alanine anions (ES-)",
        "prefix": "130423_SJA_NPALAN_1_{:03d}.raw",
        "states": {"1-": range(3, 24), "2-": range(11, 30)},
    },
    "PALAN": {
        "title": "130423 poly-DL-alanine cations (ES+)",
        "prefix": "130423_SJA_PALAN_1_{:03d}.raw",
        "states": {"1+": range(3, 17), "2+": range(11, 30)},
    },
}


def species_mz(n, charge_state):
    """The monoisotopic m/z of (Ala_n - zH)^z- or (Ala_n + zH)^z+."""
    z = int(charge_state[0])
    sign = -1.0 if charge_state[1] == "-" else +1.0
    neutral = n * RESIDUE_DA + WATER_DA
    return (neutral + sign * z * PROTON_DA) / z


def load_series(name, spec):
    """An `Experiment` for one ion series, its own conditions table plus the
    acquisitions resolved at `RAW_ROOT`."""
    conditions_path = os.path.join(DATA_DIR, name.lower(), "conditions.csv")
    toml_path = os.path.join(OUT_DIR, name.lower(), "experiment.toml")
    os.makedirs(os.path.dirname(toml_path), exist_ok=True)
    with open(toml_path, "w", encoding="utf-8", newline="\n") as fh:
        fh.write(
            "schema = 1\n"
            f'title = "{spec["title"]}"\n'
            'profile = "uw-synapt-g2"\n'
            'gas = "helium"\n'
            f'conditions = {conditions_path!r}\n'
            f'data_dir = {RAW_ROOT!r}\n'
        )
    experiment = schamp.load_experiment(toml_path)
    problems = experiment.validate(require_raw=True)
    if problems:
        raise RuntimeError(f"{name}: " + "; ".join(problems))
    return experiment


EXPERIMENTS = {name: load_series(name, spec) for name, spec in SERIES.items()}
for name, experiment in EXPERIMENTS.items():
    print(f"{name}: {len(experiment.conditions)} acquisitions, "
          f"{len(experiment.used)} kept for the regression")

In [ ]:
def windows_for(spec, spectrum, frame):
    """One window per oligomer, placed on its own measured peak where it has one."""
    windows = []
    for charge_state, ns in spec["states"].items():
        z = int(charge_state[0])
        charge = z if charge_state[1] == "+" else -z
        for n in ns:
            centre = species_mz(n, charge_state)
            windows.append(extract.MzWindow.around_measured(
                centre, spectrum, frame=frame, min_apex=MIN_APEX_COUNTS,
                fallback_width=FALLBACK_WIDTH_TH, label=f"{charge_state} n={n}",
                charge=charge, ion_mass_da=abs(charge) * centre,
            ))
    return windows


# {(series, charge_state, n): {acquisition: ATD}}
atds = {}
extraction_rows = []
for name, spec in SERIES.items():
    experiment = EXPERIMENTS[name]
    for condition in experiment.used:
        with sdk.open_readers(condition.path) as readers:
            spectrum = extract.total_spectrum(readers)
            frame = calibration.MzFrame.for_acquisition(condition.path)
            windows = windows_for(spec, spectrum, frame)
            for window, one in zip(windows, extract.extract_atds(readers, windows)):
                charge_state = window.label.split()[0]
                n = int(window.label.split("n=")[1])
                measured = "measured peak" in window.frame_note
                atds.setdefault((name, charge_state, n), {})[condition.acquisition] = one
                extraction_rows.append((name, charge_state, n, condition.acquisition,
                                        round(window.centre, 4), round(window.width, 4),
                                        round(one.total, 1), one.is_empty, measured))
    print(f"{name}: {len(experiment.used)} acquisitions")

show_table(
    ["series", "charge", "n", "acquisition", "window_mz", "window_width", "counts",
     "empty", "on_measured_peak"],
    extraction_rows, title="extracted arrival-time distributions",
)
print(f"{sum(r[-1] for r in extraction_rows)} / {len(extraction_rows)} windows on a "
      "measured peak; the rest fell back to the fixed width on the computed m/z")

## 2. Fitting: one Gaussian per arrival-time distribution

`atd.fit_gaussian` fits every distribution and reports its centroid, width and quality.
Nothing here is specific to polyalanine; the same call is made whatever the ATD holds.

In [ ]:
fits = {}
fit_rows = []
for key, per_acquisition in atds.items():
    series, charge_state, n = key
    fits[key] = {}
    for acquisition, one in per_acquisition.items():
        result = atd.fit_gaussian(one)
        fits[key][acquisition] = result
        fit_rows.append((series, charge_state, n, acquisition,
                         round(result.centre_ms, 4), round(result.sigma_ms, 4),
                         round(result.rmsd_fraction, 5), result.converged))

show_table(
    ["series", "charge", "n", "acquisition", "centre_ms", "sigma_ms", "rmsd_fraction", "converged"],
    fit_rows, title="single-Gaussian fits",
)
print(f"{sum(r[-1] for r in fit_rows)} / {len(fit_rows)} fits converged")

## 3. Regression: drift time against reciprocal drift voltage

One straight-line fit per oligomer, over the acquisitions each series' `conditions.csv`
keeps (`use = true`): arrival time in milliseconds against `1/V_drift`, giving a slope of
`L^2 / K` and an intercept `t0`, the transport time from the cell exit to the TOF
analyser (eqn 3 of the paper).

In [ ]:
regressions = {}
regression_rows = []
for key, per_acquisition in fits.items():
    series, charge_state, n = key
    experiment = EXPERIMENTS[series]
    points = [
        mobility.DriftPoint(
            acquisition=condition.acquisition,
            v_drift_v=condition.drift_voltage(PROFILE),
            drift_time_ms=per_acquisition[condition.acquisition].centre_ms,
            drift_time_ms_err=per_acquisition[condition.acquisition].centre_ms_err,
        )
        for condition in experiment.used
    ]
    line = mobility.regress(points, PROFILE)
    regressions[key] = line
    regression_rows.append((series, charge_state, n, round(line.slope_ms_v, 4),
                            round(line.slope_ms_v_err, 5), round(line.intercept_ms, 4),
                            round(line.r_squared, 6)))

show_table(
    ["series", "charge", "n", "slope_ms_V", "slope_err", "t0_ms", "r2"],
    regression_rows, title="drift-time regressions",
)

## 4. Cross sections: K, K0 and the Mason-Schamp collision cross section

`mobility.cross_section_from_regression` carries each regression through eqn (4) to a
`CrossSection`, at the mean pressure and the single temperature the series was measured
at. The error bar propagates the regression's slope error alone; `CrossSection.propagated`
says so, which is what makes it possible to read six months later what the bar does and
does not cover.

In [ ]:
cross_sections = {}
ccs_rows = []
for key, line in regressions.items():
    series, charge_state, n = key
    experiment = EXPERIMENTS[series]
    z = int(charge_state[0])
    mz = species_mz(n, charge_state)
    pressure = float(np.mean([c.pressure_torr for c in experiment.used]))
    temperature = float(np.mean([c.temperature_k for c in experiment.used]))
    xs = mobility.cross_section_from_regression(
        line, PROFILE, charge=z, ion_mass_da=z * mz, gas=experiment.gas,
        pressure_torr=pressure, temperature_k=temperature,
    )
    cross_sections[key] = xs
    ccs_rows.append((series, charge_state, n, round(mz, 4), round(xs.mobility_cm2_v_s, 4),
                     round(xs.reduced_mobility_cm2_v_s, 4), round(xs.omega_a2, 2),
                     round(xs.omega_a2_err, 2), ",".join(xs.propagated)))

show_table(
    ["series", "charge", "n", "mz", "K", "K0", "CCS_A2", "CCS_err_A2", "propagated"],
    ccs_rows, title="collision cross sections",
)

results_path = report.write_table(
    os.path.join(OUT_DIR, "polyalanine-ccs.csv"),
    ["series", "charge_state", "n", "mz", "K_cm2_per_Vs", "K0_cm2_per_Vs", "CCS_A2",
     "CCS_err_A2", "propagated"],
    ccs_rows,
    comment=(
        "Poly-DL-alanine collision cross sections in helium, from "
        "examples/polyalanine-walkthrough.ipynb.\n"
        "Extraction, fitting, regression and the Mason-Schamp conversion are schamp's "
        "own; nothing here\nis typed in except the species definitions and the window "
        "parameters in the extraction section's first code cell."
    ),
)
print("wrote", results_path)

## The figure

Three panels, the same three the paper draws from this series: a mobiligram heat map
(one acquisition, every m/z, drift time against intensity), drift time against `1/V`
per oligomer with the fitted lines, and collision cross section against m/z for all
four ion series. Colours and the exact axis ranges are a notebook's choice, not a
result -- `examples/README.md` says how this rendering differs from the published one.

In [ ]:
report.use_headless_matplotlib()
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(11.0, 3.6))
axes = [fig.add_subplot(1, 3, i + 1) for i in range(3)]

# Panel A: the heat map, one acquisition, a contiguous grid over the whole mass range.
heat_condition = min(EXPERIMENTS["NPALAN"].used, key=lambda c: c.drift_voltage(PROFILE))
grid = extract.contiguous_windows(200.0, 1700.0, 255)
with sdk.open_readers(heat_condition.path) as readers:
    centres, drift_ms, intensity = extract.mobility_map(readers, grid)
z_norm = np.sqrt(intensity / intensity.max())
axes[0].imshow(
    z_norm, aspect="auto", origin="lower", cmap="viridis",
    extent=(centres.min(), centres.max(), drift_ms.min(), drift_ms.max()),
)
axes[0].set_xlabel("m/z")
axes[0].set_ylabel("drift time / ms")
axes[0].set_title(f"{os.path.basename(heat_condition.path)}\n(lowest kept drift voltage)", fontsize=8)

# Panel B: the anion series, one line and its own ten points per oligomer -- a fan of
# near-parallel lines, one per n, which is what eqn (3) predicts for a series that
# differs only in mass.
colour = {"1-": "#943126", "2-": "#1b4f72", "1+": "#7d3c98", "2+": "#1e8449"}
experiment = EXPERIMENTS["NPALAN"]
inverse_v = np.array([1.0 / c.drift_voltage(PROFILE) for c in experiment.used])
xs_fit = np.linspace(inverse_v.min(), inverse_v.max(), 2)
for charge_state in ("1-", "2-"):
    keys = [k for k in regressions if k[0] == "NPALAN" and k[1] == charge_state]
    for i, key in enumerate(keys):
        centre_ms = np.array([fits[key][c.acquisition].centre_ms for c in experiment.used])
        axes[1].plot(inverse_v, centre_ms, "o", ms=1.5, color=colour[charge_state], alpha=0.5)
        line = regressions[key]
        axes[1].plot(
            xs_fit, line.intercept_ms + line.slope_ms_v * xs_fit, "-", lw=0.5,
            color=colour[charge_state], alpha=0.7,
            label=charge_state if i == 0 else None,
        )
axes[1].set_xlabel("1 / V$_{drift}$ / V$^{-1}$")
axes[1].set_ylabel("drift time / ms")
axes[1].legend(fontsize=7)

# Panel C: cross section against m/z, all four series together.
marker = {"1-": "o", "2-": "o", "1+": "s", "2+": "s"}
for charge_state in ("1-", "2-", "1+", "2+"):
    keys = [k for k in cross_sections if k[1] == charge_state]
    if not keys:
        continue
    mzs = [species_mz(k[2], charge_state) for k in keys]
    omegas = [cross_sections[k].omega_a2 for k in keys]
    errs = [cross_sections[k].omega_a2_err for k in keys]
    axes[2].errorbar(mzs, omegas, yerr=errs, fmt=marker[charge_state], ms=3,
                     color=colour[charge_state], label=charge_state)
axes[2].set_xlabel("m/z")
axes[2].set_ylabel(r"$\Omega$ (He) / $\AA^2$")
axes[2].legend(fontsize=7)

fig.tight_layout()
figure_path = os.path.join(OUT_DIR, "polyalanine-figure.png")
fig.savefig(figure_path)
plt.show()
print("wrote", figure_path)

## Pointing this at your own series

Everything above generalises past polyalanine by changing four things, all in the cells
above rather than in `schamp` itself:

1. **The species definitions**, in the extraction section's first code cell:
   `species_mz` and the `states` ranges in `SERIES`. Any m/z series works; a single
   analyte at one charge state is `states = {"1-": range(1, 2)}` with `species_mz`
   returning its m/z regardless of `n`.
2. **`examples/data/<series>/conditions.csv`**, one row per acquisition: its pressure,
   temperature, and whether the regression should use it. Write your own in this shape
   (`examples/README.md` has the column list) rather than editing this notebook's data.
3. **Where the acquisitions live**: `$SCHAMP_EXAMPLE_RAW`, pointed at the directory
   holding them -- no lab checkout needed once that is set.
4. **The instrument profile**, `schamp.load_profile("uw-synapt-g2")`. A different copy
   of the drift cell, or a different gas, is a different profile
   (`src/schamp/data/profiles/uw-synapt-g2.toml` is annotated as the format
   specification) or a different `gas =` in `experiment.toml`.

`MIN_APEX_COUNTS` and `FALLBACK_WIDTH_TH` are the two numbers this series needed tuning:
the first is comfortably above this series' baseline and below its weakest real peak,
and the second only matters for a species too weak to measure, which never happens on
this series. `notes/extraction.md` (lab record) has how both were chosen.